<a href="https://colab.research.google.com/github/SabatoCom24212/podcast-tcom/blob/main/podcast_tcom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install -q git+https://github.com/resemble-ai/chatterbox.git
!pip install -q pydub groq datasets soundfile nest_asyncio

import chatterbox
print(chatterbox.__version__)  # debe ser > 0.1.7

In [ ]:
import torch, torchvision
print(torch.__version__)         # 2.6.0+cu124
print(torchvision.__version__)   # 0.21.0+cu124
print(torch.cuda.is_available()) # True

In [ ]:
!pip install -q --force-reinstall "datasets==2.19.2" "fsspec==2025.3.0"

import datasets
print(datasets.__version__)  # debe ser 2.19.2

In [ ]:
from datasets import load_dataset
ds = load_dataset(
    "ylacombe/google-argentinian-spanish",
    "female", split="train", trust_remote_code=True
)
# Intentar leer el primer audio
sample = ds[0]
print(sample["speaker_id"])
print(sample["audio"]["sampling_rate"])

In [ ]:
"""
Arquitectura del pipeline:
  guion.txt
      │
      ▼
  leer_guion()
      │
      ▼
  preprocesar_texto()          ← CAPA 1: normalización + fonetización rioplatense
      │                           Se ejecuta de forma IDÉNTICA en inferencia.
      ▼
  fragmentar()                 ← chunking por límites de oración
      │
      ▼
  ChatterboxTTS.generate()     ← síntesis con voice prompt argentino
      │
      ▼
  mezcla + export → podcast_final.mp3
"""

import re
import os
import torch
import warnings
import tempfile
import logging
import traceback

import torchaudio as ta
import nest_asyncio
from pydub import AudioSegment

nest_asyncio.apply()
warnings.filterwarnings("ignore")
logging.getLogger("chatterbox").setLevel(logging.ERROR)
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["TRANSFORMERS_NO_ADVICE"] = "1"


# =============================================================================
# CONFIGURACIÓN GLOBAL
# =============================================================================

GROQ_API_KEY    = "TU_API_KEY_AQUI"    # Solo necesario si MODO_GUION = "groq"
GROQ_MODEL      = "llama-3.3-70b-versatile"
ARCHIVO_GUION   = "guion_dialogo.txt"  # Formato: "Clara: ..." / "Leo: ..."
ARCHIVO_MUSICA  = "musica.mp3"

# "archivo" → lee el guion directamente (control total, Groq no se usa)
# "groq"    → usa Groq para generar el diálogo a partir del texto del archivo
MODO_GUION      = "archivo"

SPEAKER_ID_CLARA = 9697
SPEAKER_ID_LEO   = 7508

# Hiperparámetros de síntesis por locutor
CFG_WEIGHT      = 0.5   # más anclaje al prompt, menos alucinaciones
EXG_CLARA       = 0.65  # menos varianza, generación más estable
EXG_LEO         = 0.55

os.environ["GROQ_API_KEY"] = GROQ_API_KEY
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo: {DEVICE.upper()}")

SR_MODEL = None  # Se asigna tras cargar el modelo TTS


# =============================================================================
# CAPA 1 — PREPROCESAMIENTO DE TEXTO (NLP + FONETIZACIÓN RIOPLATENSE)
# =============================================================================
# Esta capa debe ejecutarse de forma IDÉNTICA en entrenamiento e inferencia
# para garantizar consistencia entre el texto visto durante fine-tuning y
# el texto enviado al modelo en producción.
#
# Decisiones de diseño:
#
# 1. PLACEHOLDERS UNICODE (área de uso privado, U+E001–U+E004):
#    Se usan para \"congelar\" grafemas que no deben tocarse (conjunción 'y',
#    'y' en coda de diptongo) antes de aplicar las sustituciones globales.
#    Esto evita colisiones con el motor de regex posterior.
#
# 2. YEÍSMO REHILADO [ʃ]/[ʒ]:
#    - ll (todas las capitalizaciones) → sh
#    - y consonántica (onset silábico) → sh
#    - y conjunción ('Pedro y María')  → INTOCABLE (marcada con U+E001/E002)
#    - y en coda de diptongo (hoy, hay, rey, muy) → INTOCABLE (U+E003/E004)
#    - 'mayor', 'ayer' → 'mashor', 'asher': correcto en fonética rioplatense,
#      ya que la 'y' intervocálica en onset SÍ es rehilada en Buenos Aires.
#
# 3. GLOSARIO IT:
#    Sustituye términos técnicos por formas fonéticas antes de la tokenización
#    para evitar que el modelo los pronuncie al inglés o en modo neutro.
#    Se aplica ANTES del motor de yeísmo para evitar interferencias.

# Placeholders en el área de uso privado del Unicode (nunca aparecen en texto normal)
_MARCA_CONJ_MIN  = '\\uE001'   # conjunción 'y' minúscula
_MARCA_CONJ_MAY  = '\\uE002'   # conjunción 'Y' mayúscula
_MARCA_DIPT_MIN  = '\\uE003'   # 'y' en coda de diptongo, minúscula (hoy, hay, rey)
_MARCA_DIPT_MAY  = '\\uE004'   # ídem, mayúscula

# Patrón auxiliar (letra española, incluyendo tildes y ñ)
_LETRA = r'[A-ZÁÉÍÓÚÜÑa-záéíóúüñ]'
_VOCAL = r'[aeiouáéíóúüAEIOUÁÉÍÓÚÜ]'

# Glosario IT: tuplas (patrón_regex, reemplazo_fonético)
# Orden importa: los más específicos van primero.
_GLOSARIO_IT = [
    (r'(?i)\\bPull\\s+Requests?\\b',       'pul ricuést'),
    (r'(?i)\\bConventional\\s+Commits\\b', 'convénshonal comíts'),
    (r'(?i)\\bsoftware\\b',               'sófwer'),
    (r'(?i)\\bhardware\\b',               'járdwer'),
    (r'(?i)\\bREADME\\b',                 'rídmi'),
    (r'(?i)\\bgit\\b',                    'guit'),
    (r'(?i)\\bcommit\\b',                 'comít'),
    (r'(?i)\\bbranch\\b',                 'branch'),   # se pronuncia aceptablemente
    (r'(?i)\\bpipeline\\b',               'páipshain'),
    (r'(?i)\\bframework\\b',              'fréimwork'),
    (r'(?i)\\bbackend\\b',                'bákend'),
    (r'(?i)\\bfrontend\\b',               'frónend'),
    (r'(?i)\\bdeployment\\b',             'dipshóiment'),
    (r'(?i)\\brelease\\b',                'rilís'),
]


def _aplicar_glosario_it(texto: str) -> str:
    \"\"\"Reemplaza términos técnicos por su forma fonética rioplatense.\"\"\"
    for patron, reemplazo in _GLOSARIO_IT:
        texto = re.sub(patron, reemplazo, texto)
    return texto


def _fonetizar_yeismo(texto: str) -> str:
    \"\"\"
    Motor de yeísmo rehilado porteño. Opera en cinco pasos:
      1. Proteger conjunción 'y/Y' (átona, no adyacente a letras).
      2. Proteger 'y/Y' en coda de diptongo (hoy, hay, rey, Uruguay...).
      3. ll/LL/Ll → sh/SH/Sh.
      4. y/Y consonántica restante → sh/Sh.
      5. Restaurar placeholders.
    \"\"\"
    # Paso 1: conjunción átona (no precedida ni seguida de letra)
    texto = re.sub(rf'(?<!{_LETRA[1:]})(y)(?!{_LETRA[1:]})', _MARCA_CONJ_MIN, texto)
    texto = re.sub(rf'(?<!{_LETRA[1:]})(Y)(?!{_LETRA[1:]})', _MARCA_CONJ_MAY, texto)

    # Paso 2: 'y' en coda silábica (vocal + y + no-letra o fin de cadena)
    texto = re.sub(rf'(?<={_VOCAL[1:]})(y)(?=[^{_LETRA[2:]}]|\\Z)', _MARCA_DIPT_MIN, texto)
    texto = re.sub(rf'(?<={_VOCAL[1:]})(Y)(?=[^{_LETRA[2:]}]|\\Z)', _MARCA_DIPT_MAY, texto)

    # Paso 3: dígrafos ll
    texto = re.sub(r'll', 'sh',  texto)
    texto = re.sub(r'Ll', 'Sh',  texto)
    texto = re.sub(r'LL', 'SH',  texto)

    # Paso 4: y consonántica restante (onset silábico)
    texto = re.sub(r'y', 'sh',   texto)
    texto = re.sub(r'Y', 'Sh',   texto)

    # Paso 5: restaurar
    texto = (texto
             .replace(_MARCA_DIPT_MIN, 'y').replace(_MARCA_DIPT_MAY, 'Y')
             .replace(_MARCA_CONJ_MIN, 'y').replace(_MARCA_CONJ_MAY, 'Y'))

    return texto


def preprocesar_texto(texto: str) -> str:
    \"\"\"
    Punto de entrada único de la Capa 1.
    Aplicar en TODOS los textos antes de enviarlos al modelo TTS,
    tanto en fine-tuning como en inferencia.

    Orden de operaciones:
      1. Limpieza de markup (HTML, asteriscos de markdown).
      2. Glosario IT (sustituye anglicismos antes del motor fonético).
      3. Motor de yeísmo rehilado.
    \"\"\"
    # 1. Limpieza de markup
    texto = re.sub(r'<[^>]+>', '', texto)       # etiquetas HTML/XML
    texto = re.sub(r'\\*+',     '', texto)        # negritas/itálicas markdown
    texto = texto.strip()

    # 2. Glosario IT (antes del yeísmo para evitar que 'sh' de 'sh' en 'sófwer' se duplique)
    texto = _aplicar_glosario_it(texto)

    # 3. Motor de yeísmo rehilado
    texto = _fonetizar_yeismo(texto)

    return texto


# =============================================================================
# CAPA 2 — CONFIGURACIÓN DE FINE-TUNING (FREEZING + LoRA + SCHEDULER)
# =============================================================================
# Esta sección no modifica el flujo de inferencia actual pero documenta
# y expone los hiperparámetros de entrenamiento para cuando se realice
# fine-tuning sobre el dataset argentino.
#
# Estrategia anti-Catastrophic Forgetting:
#   - Se congela el encoder de texto y el predictor de duración (capas
#     que codifican conocimiento fonológico general del español neutro).
#   - Se deja abierto solo el decoder y el módulo de proyección acústica
#     (capas responsables de la prosodia y el timbre del acento objetivo).
#   - Con LoRA (r=32, alpha=64) sobre las capas de atención del decoder
#     se logra adaptación eficiente con datasets pequeños (~3h).
#
# Justificación de hiperparámetros:
#   - lr=5e-5: conservador para no borrar el prior fonológico del modelo.
#   - linear decay scheduler: evita overshooting al final del entrenamiento.
#   - batch_size=8: equilibrio entre estabilidad de gradiente y VRAM (~16GB).
#   - warmup_steps=100: permite que LoRA se inicialice antes del decay.

FINETUNING_CONFIG = {
    # ── Optimizador ────────────────────────────────────────────
    \"learning_rate\":    5e-5,
    \"scheduler\":        \"linear\",       # caída lineal desde lr hasta 0
    \"warmup_steps\":     100,
    \"batch_size\":       8,              # para ~3h de audio en T4/A100
    \"max_epochs\":       30,
    \"gradient_clip\":    1.0,

    # ── LoRA (si el backend lo soporta) ────────────────────────
    # Rank bajo: adapta el acento sin olvidar la fonología base.
    # alpha = 2*r es la heurística estándar para síntesis de voz.
    \"lora_r\":           32,
    \"lora_alpha\":       64,
    \"lora_dropout\":     0.05,
    \"lora_target_modules\": [\"attn_q\", \"attn_k\", \"attn_v\", \"attn_out\"],  # solo atención del decoder

    # ── Capas congeladas (freeze) ───────────────────────────────
    # Congelar: encoder de texto + predictor de duración
    # Descongelar: decoder + proyección acústica
    \"freeze_modules\": [
        \"text_encoder\",         # codificación fonológica del español neutro
        \"duration_predictor\",   # ritmo base — se reaprendería con sesgo caribeño
    ],
    \"trainable_modules\": [
        \"decoder\",              # donde vive la prosodia y el acento
        \"acoustic_projection\",  # proyección al espacio mel
    ],
}


def aplicar_freeze(model, config: dict = FINETUNING_CONFIG) -> None:
    \"\"\"
    Congela los módulos listados en 'freeze_modules' y descongela
    los listados en 'trainable_modules'.

    Uso (en el script de fine-tuning, no en inferencia):
        modelo = TTS.from_pretrained(...)
        aplicar_freeze(modelo)
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, modelo.parameters()),
            lr=FINETUNING_CONFIG[\"learning_rate\"]
        )
    \"\"\"
    # Primero congelar todo
    for param in model.parameters():
        param.requires_grad = False

    # Luego descongelar solo los módulos objetivo
    for nombre, modulo in model.named_modules():
        for patron in config[\"trainable_modules\"]:
            if patron in nombre:
                for param in modulo.parameters():
                    param.requires_grad = True

    entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total       = sum(p.numel() for p in model.parameters())
    print(f\"Parámetros entrenables: {entrenables:,} / {total:,} \"
          f\"({100 * entrenables / total:.1f}%)\")


def construir_scheduler(optimizer, config: dict = FINETUNING_CONFIG, total_steps: int = 1000):
    \"\"\"
    Retorna un scheduler de caída lineal con warmup.
    Importar desde transformers en el entorno de fine-tuning.
    \"\"\"
    try:
        from transformers import get_linear_schedule_with_warmup
        return get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=config[\"warmup_steps\"],
            num_training_steps=total_steps,
        )
    except ImportError:
        print(\"Instalar transformers para usar el scheduler: pip install transformers\")
        return None


# =============================================================================
# CAPA 3 — PIPELINE DE INFERENCIA
# =============================================================================

def tensor_a_pydub(tensor_wav: torch.Tensor) -> AudioSegment:
    \"\"\"Convierte un tensor WAV a un objeto AudioSegment de pydub.\"\"\"
    with tempfile.NamedTemporaryFile(suffix=\".wav\", delete=False) as tmp:
        ta.save(tmp.name, tensor_wav.cpu(), SR_MODEL)
        return AudioSegment.from_wav(tmp.name)


def fragmentar(texto: str, max_chars: int = 100) -> list[str]:
    \"\"\"
    Divide el texto en fragmentos respetando límites naturales del habla.
    Límite reducido a 100 chars (vs 140 anterior) para evitar que el modelo
    TTS supere su ventana de contexto y genere audio degradado.

    Jerarquía de corte:
      1. Límites de oración  (.  !  ?)
      2. Cláusulas separadas por coma o punto y coma
      3. Corte duro por espacio si no hay puntuación disponible
    \"\"\"
    # Paso 1: partir por fin de oración
    oraciones = re.split(r'(?<=[.!?])\\s+', texto)

    fragmentos = []
    for oracion in oraciones:
        if len(oracion) <= max_chars:
            fragmentos.append(oracion)
        else:
            # Paso 2: partir por coma / punto y coma dentro de la oración
            clausulas = re.split(r'(?<=[,;])\\s+', oracion)
            actual = \"\"
            for c in clausulas:
                if len(actual) + len(c) <= max_chars:
                    actual = (actual + \" \" + c).strip()
                else:
                    if actual:
                        fragmentos.append(actual)
                    # Paso 3: corte duro si la cláusula sola excede el límite
                    if len(c) > max_chars:
                        palabras = c.split()
                        actual = \"\"
                        for p in palabras:
                            if len(actual) + len(p) + 1 <= max_chars:
                                actual = (actual + \" \" + p).strip()
                            else:
                                if actual:
                                    fragmentos.append(actual)
                                actual = p
                    else:
                        actual = c
            if actual:
                fragmentos.append(actual)

    return [f for f in fragmentos if f.strip()] or [texto]
